# Initial Work to Digitize Calorimeter Hits

In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import uproot
import sys

import atlasify as atl
from particle import Particle
atl.ATLAS = "ColliderML"

sys.path.append("../")
# from utils import load_root_file, load_hepmc_event
from edm4hep_utils import build_calo_df, build_particle_df, build_tracker_df, load_edm4hep_file
from clustering_metrics import evaluate_clustering, plot_clustering_metrics

from edm4hep_utils import pixel_readouts, strip_readouts
all_tracker_readouts = pixel_readouts + strip_readouts

## Roadmap

1. Load in ecal barrel calorimeter hits
2. Structure a digitisation function

# Loading

In [2]:
base_dir = "/global/cfs/cdirs/m4958/data/ColliderML/simulation/single_particle_pion_10GeV"
event_num = 0

## 1. Load in edm4hep file

In [3]:
low_threshold_edm4hep_file = f"{base_dir}/single_run_test_low_threshold/edm4hep.root"

low_threshold_event = load_edm4hep_file(low_threshold_edm4hep_file, event_num=event_num)

In [4]:
low_threshold_event.keys()

dict_keys(['tracker_df', 'calo_hits_df', 'calo_contrib_df', 'particles_df', 'parents_df', 'daughters_df'])

In [5]:
low_tracker_df = low_threshold_event["tracker_df"]
low_tracker_df.columns

Index(['cellID', 'EDep', 'time', 'pathLength', 'quality', 'x', 'y', 'z', 'px',
       'py', 'pz', 'particle_id', 'r', 'R', 'phi', 'theta', 'eta', 'pt',
       'detector'],
      dtype='object')

In [6]:
low_parents_df = low_threshold_event["parents_df"]

low_daughters_df = low_threshold_event["daughters_df"]

low_particles_df = low_threshold_event["particles_df"]

# Create a column from the index
low_particles_df.columns

Index(['PDG', 'generatorStatus', 'simulatorStatus', 'charge', 'time', 'mass',
       'vx', 'vy', 'vz', 'px', 'py', 'pz', 'endpoint_x', 'endpoint_y',
       'endpoint_z', 'parents_begin', 'parents_end', 'daughters_begin',
       'daughters_end', 'pt', 'p', 'eta', 'phi'],
      dtype='object')

In [7]:
low_calo_hits_df = low_threshold_event["calo_hits_df"]
low_calo_hits_df.columns

Index(['cellID', 'energy', 'x', 'y', 'z', 'contribution_begin',
       'contribution_end', 'r', 'R', 'phi', 'theta', 'eta', 'detector'],
      dtype='object')

In [8]:
low_calo_contrib_df = low_threshold_event["calo_contrib_df"]
low_calo_contrib_df.columns


Index(['PDG', 'energy', 'time', 'step_x', 'step_y', 'step_z', 'particle_id',
       'x', 'y', 'z', 'detector'],
      dtype='object')

## Visualize hits in one event in all three subdetectors

In [148]:
low_particles_df

,PDG,generatorStatus,simulatorStatus,charge,time,mass,vx,vy,vz,px,...,endpoint_y,endpoint_z,parents_begin,parents_end,daughters_begin,daughters_end,pt,p,eta,phi
0,211,1,150994944,1.0,0.000000,0.139570,0.000000,0.000000,0.000000,7.314463,...,215.503582,-1053.953214,0,0,0,25,7.509689,10.000000,-0.793412,0.228517
1,11,0,1493172224,-1.0,2.822778,0.000511,622.399472,128.314567,-558.626150,0.002685,...,131.401366,-581.983508,0,1,25,25,0.003212,0.003374,-0.316664,0.580828
2,-211,0,1157627904,-1.0,5.325320,0.139570,1178.951708,215.503582,-1053.953214,2.335005,...,306.682803,-1544.734026,1,2,25,25,2.396004,3.456059,-0.909041,0.226132
3,211,0,1157627904,1.0,5.325320,0.139570,1178.951708,215.503582,-1053.953214,1.061720,...,205.171274,-1123.123427,2,3,25,38,1.067991,1.306458,-0.656411,-0.108426
4,2112,0,1157627904,0.0,5.325320,0.939565,1178.951708,215.503582,-1053.953214,0.548697,...,118.435793,-1018.249183,3,4,38,38,0.625221,0.635842,0.184068,-0.499952
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,11,0,1493172224,-1.0,21.051262,0.000511,1180.984584,55.996198,-1338.326321,-0.000095,...,55.768974,-1337.425366,98,99,103,103,0.000601,0.001560,1.607024,-1.729773
100,11,0,1493172224,-1.0,32.093857,0.000511,40.244411,1169.299437,-687.859030,-0.000683,...,1171.264470,-686.715277,99,100,103,103,0.002097,0.002320,0.456166,1.902523
101,11,0,1493172224,-1.0,32.273067,0.000511,-8.106250,1192.619432,-685.637901,-0.002068,...,1192.132471,-685.777066,100,101,103,103,0.002146,0.002212,-0.247636,2.871881
102,11,0,1493172224,-1.0,2371.792969,0.000511,354.308557,1126.995840,-1355.059080,0.000104,...,1124.697051,-1357.129397,101,102,103,103,0.001653,0.003156,-1.262515,-1.507632


In [151]:
low_tracker_df

,cellID,EDep,time,pathLength,quality,x,y,z,px,py,pz,particle_id,r,R,phi,theta,eta,pt,detector
0,10997633253382,0.000046,0.143050,0.167380,0,31.313939,7.240239,-28.260180,7.319360,1.682778,-6.602227,0,32.140066,42.797448,0.227222,2.292046,-0.793388,7.510312,PixelBarrelReadout
1,9687668228358,0.000042,0.147953,0.173594,0,32.448207,7.500762,-29.283262,7.319657,1.681203,-6.601789,0,33.303868,44.347007,0.227171,2.292043,-0.793383,7.510248,PixelBarrelReadout
2,249177578012950,0.000062,0.302190,0.167145,0,66.250578,15.215565,-59.748868,7.324624,1.661593,-6.599766,0,67.975382,90.501822,0.225752,2.291875,-0.793160,7.510726,PixelBarrelReadout
3,11269390533158,0.000046,0.508585,0.168126,0,111.588953,25.423328,-100.590262,7.329116,1.636143,-6.599785,0,114.448417,152.370735,0.224007,2.291839,-0.793111,7.509521,PixelBarrelReadout
4,43017318957878,0.000055,0.758126,0.168246,0,166.447468,37.540911,-149.978514,7.334657,1.602737,-6.600558,0,170.628485,227.173137,0.221830,2.291874,-0.793159,7.507726,PixelBarrelReadout
5,9687668228358,0.000021,0.147912,0.078876,1073741824,32.394304,7.491571,-29.282581,-0.000169,-0.000143,-0.000758,0,33.249279,44.305576,0.227267,2.292845,-0.794451,0.000222,PixelBarrelReadout
6,4189757767942,0.000065,0.165217,0.233655,1073741824,32.965128,6.582458,-33.248635,0.000431,0.000016,-0.000167,0,33.615895,47.281076,0.197087,2.350702,-0.873627,0.000431,PixelBarrelReadout
7,71565940542144777,0.000071,1.168205,0.271665,0,256.684054,56.904351,-231.099534,7.345233,1.548243,-6.598343,0,262.915973,350.045431,0.218162,2.291880,-0.793166,7.506631,ShortStripBarrelReadout
8,351633267950105,0.000107,1.597078,0.268385,0,351.246436,76.453148,-315.975668,7.355785,1.491108,-6.597908,0,359.470642,478.601886,0.214320,2.291889,-0.793178,7.505397,ShortStripBarrelReadout
9,71951096029250345,0.000072,2.211591,0.267508,0,486.984105,103.195472,-437.601316,7.371936,1.409538,-6.595852,0,497.797975,662.795395,0.208818,2.291929,-0.793232,7.505480,ShortStripBarrelReadout


In [153]:
low_calo_contrib_df

,PDG,energy,time,step_x,step_y,step_z,particle_id,x,y,z,detector
0,0,2.068670e-05,21.231413,0.0,0.0,0.0,24,1292.800049,219.300003,-1759.500000,ECalBarrelCollection
1,0,5.017930e-07,39.168983,0.0,0.0,0.0,24,1277.650024,224.399994,-1764.599976,ECalBarrelCollection
2,0,2.366167e-04,5473.687500,0.0,0.0,0.0,24,1356.866821,313.623596,-1892.099976,ECalBarrelCollection
3,0,1.360825e-04,5473.688965,0.0,0.0,0.0,24,1356.866821,313.623596,-1892.099976,ECalBarrelCollection
4,0,1.016068e-04,5473.690430,0.0,0.0,0.0,24,1356.866821,313.623596,-1892.099976,ECalBarrelCollection
...,...,...,...,...,...,...,...,...,...,...,...
1913,0,5.565418e-07,1064.158691,0.0,0.0,0.0,20,11.385399,364.788116,-4208.500000,HCalEndcapCollection
1914,0,3.059080e-08,8456.931641,0.0,0.0,0.0,20,335.524567,-158.503189,-4259.500000,HCalEndcapCollection
1915,0,1.876329e-09,8465.379883,0.0,0.0,0.0,20,335.524567,-158.503189,-4259.500000,HCalEndcapCollection
1916,0,6.513178e-05,62.221786,0.0,0.0,0.0,30,-411.769806,-71.032570,-3749.500000,HCalEndcapCollection


In [11]:
low_calo_hits_df

,cellID,energy,x,y,z,contribution_begin,contribution_end,r,R,phi,theta,eta,detector
0,18349635391436681232,2.068670e-05,1292.800049,219.300003,-1759.500000,0,1,1311.268311,2194.371094,0.168032,2.501138,-1.103700,ECalBarrelCollection
1,18349353920754839568,5.017930e-07,1277.650024,224.399994,-1764.599976,1,2,1297.206665,2190.104492,0.173862,2.507678,-1.114693,ECalBarrelCollection
2,18342598139062315024,1.239383e-03,1356.866821,313.623596,-1892.099976,2,37,1392.640381,2349.359375,0.227149,2.507093,-1.113706,ECalBarrelCollection
3,18347383196485871632,1.050777e-06,1271.361694,256.125427,-1805.400024,37,38,1296.904297,2222.932861,0.198797,2.518659,-1.133373,ECalBarrelCollection
4,18375249365209204752,5.508685e-07,-422.123810,1219.001099,-1300.500000,38,39,1290.020264,1831.789429,1.904159,2.360240,-0.887106,ECalBarrelCollection
...,...,...,...,...,...,...,...,...,...,...,...,...,...
592,18431262980061622291,1.629226e-04,1698.500000,210.000000,-1650.000000,1909,1912,1711.432861,2377.288818,0.123014,2.337921,-0.855762,HCalBarrelCollection
593,3377708317147668,3.306568e-06,11.385399,364.788116,-4208.500000,1912,1914,364.965759,4224.295410,1.539595,3.055088,-3.140081,HCalEndcapCollection
594,18446181072223240724,3.246713e-08,335.524567,-158.503189,-4259.500000,1914,1916,371.079498,4275.633301,-0.441328,3.054694,-3.135530,HCalEndcapCollection
595,18445336754666963476,6.513178e-05,-411.769806,-71.032570,-3749.500000,1916,1917,417.851654,3772.711182,-2.970768,3.030609,-2.890489,HCalEndcapCollection


In [154]:
def plot_event_particles(tracker_df, calo_contrib_df, calo_hits_df=None, figsize=(10, 10)):

    fig = plt.figure(figsize=figsize)
    
    # Combine the unique particle IDs from both dataframes
    unique_particles = set(tracker_df.particle_id.unique()) | set(calo_contrib_df.particle_id.unique())
    
    colors = plt.cm.rainbow(np.linspace(0, 1, len(unique_particles)))
    
    # Determine the maximum energy for normalization
    max_energy = calo_hits_df.energy.max() if calo_hits_df is not None else calo_contrib_df.energy.max()
    
    for particle, color in zip(unique_particles, colors):
        # Plot calorimeter hits
        particle_data = calo_contrib_df[calo_contrib_df.particle_id == particle]
        if not particle_data.empty:  # Only plot if there's data
            alphas = np.sqrt(particle_data.energy / max_energy)
            plt.scatter(particle_data.x, particle_data.y, color=color, marker="o", alpha=alphas)
        
        # Plot tracker hits
        tracker_data = tracker_df[tracker_df.particle_id == particle]
        if not tracker_data.empty:
            plt.scatter(tracker_data.x, tracker_data.y, color=color, marker="x")

In [ ]:
plot_event_particles(low_tracker_df, low_calo_contrib_df)

## ECal Digitisation

In [14]:
hits = low_calo_hits_df[low_calo_hits_df.detector == "ECalBarrelCollection"]
contribs = low_calo_contrib_df[low_calo_contrib_df.detector == "ECalBarrelCollection"]

In [ ]:
def digitise_ecal_hits(hits, contribs):
    """
    Digitise the ECal hits and contributions.

    Args:
        hits: pd.DataFrame
            The hits dataframe.
        contribs: pd.DataFrame
            The contributions dataframe.

    Returns:
        pd.DataFrame
            The digitised hits dataframe.
    """

    # ------------- BASIC DIGITISATION STEPS -------------

    # Threshold effects (timing and energy)



    # Dead channels



    # Layerwise miscalibration

    

    # ------------- REALISIC DIGITISATION -------------

    
    # Calculate #e-h pairs


    # Fluctuate it by Poisson
    

    # Limited electronics dynamic range


    # Add electronics noise


    return hits

